# QUELL — Multi-seed variance (Edge headline)

Edge'de LLM (0.929) vs RF (0.920) farki gercek mi, gurultu mu? Her modeli birkac seedle yeniden egitip **macro-F1 ortalama +/- std** verir. LLM 4-bit QLoRA (headline ile ayni ayar), RF+XGB tam cok-class.

Her seed sonrasi JSON'a kaydeder (cokerse devam eder). 5 seed ~1.5 saat. Hizli deneme: `SEEDS=[42,1]`.
Sonda ORTALAMA +/- STD tablosunu ve 'LLM-RF fark' rowsini paylas.

In [ ]:
# ============ QUELL — Multi-seed variance (Edge headline: LLM vs RF vs XGB) ============
# Goal: is the Edge LLM 0.929 vs RF 0.920 gap REAL or just noise? Each model is
# retrain with several seeds and report macro-F1 mean +/- std. A Q1 reviewer expects this.
# Not: the balanced training subset varies with the seed + LLM init varies with the seed -> real variance.
import os, subprocess
try:
    _o=subprocess.check_output("nvidia-smi --query-gpu=index,memory.free --format=csv,noheader,nounits",shell=True,text=True)
    _f=[(int(x.split(",")[0]),int(x.split(",")[1])) for x in _o.strip().splitlines()]
    _b=max(_f,key=lambda t:t[1]); os.environ["CUDA_VISIBLE_DEVICES"]=str(_b[0])
    os.environ["PYTORCH_CUDA_ALLOC_CONF"]="expandable_segments:True"
    print("selected GPU:",_b[0],"| free(MiB):",_f,flush=True)
except Exception as e: print("GPU secim atlandi:",e)
import json, time, sys, gc
from pathlib import Path
import numpy as np, pandas as pd
from pandas.api.types import is_numeric_dtype
for pk in ["transformers","peft","accelerate","bitsandbytes","xgboost"]:
    try: __import__(pk)
    except Exception: subprocess.run([sys.executable,"-m","pip","install","-q",pk])
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, BitsAndBytesConfig, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
import xgboost as xgb

# ===== AYARLAR =====
DATASET="edge_iiotset"; MODEL="Qwen/Qwen2.5-1.5B"
SEEDS=[42,1,2,3,4]                # 5 seeds (42 is the headline). For a quick trial use [42,1].
TRAIN_CAP=2000; EPOCHS=2          # SAME setting as the headline (0.929) -> comparable
# ===================
ROOT=Path.home()/"quell-edge-llm-ids"; PROC=ROOT/"data"/"processed"; SPL=ROOT/"splits"; RES=ROOT/"results"
rep=json.load(open(RES/"split_report.json")); meta=rep[DATASET]
label=meta["label_col"]; group=meta.get("group_col"); tcol=meta.get("time_col")
LABELISH={"label","attack","attack_type","attack_label","type","class","category","marker","__label__"}
df=pd.read_parquet(PROC/f"{DATASET}.parquet").reset_index(drop=True)
sp=np.load(SPL/f"{DATASET}_split.npz"); tr_idx,te_idx=sp["train"],sp["test"]
drop=set([label])|set(meta.get("leaky_candidates",[]))
if group: drop.add(group)
if tcol: drop.add(tcol)
for c in df.columns:
    if c!=label and c.lower() in LABELISH: drop.add(c)
feats=[c for c in df.columns if c not in drop]
y_all=df[label].astype(str).values
classes_all=sorted(pd.unique(y_all).tolist())
MAX_LEN=256 if len(feats)<=64 else 512
BF16=torch.cuda.is_available() and torch.cuda.is_bf16_supported()

# numeric matrix for RF/XGB
Xdf=df[feats].copy()
for c in Xdf.columns:
    if not is_numeric_dtype(Xdf[c]): Xdf[c]=pd.factorize(Xdf[c])[0]
X=np.nan_to_num(Xdf.values.astype("float32"))
# text for the LLM
def row_to_text(r):
    parts=[]
    for c in feats:
        v=r[c]
        if isinstance(v,(float,np.floating)): v=round(float(v),4)
        parts.append(f"{c}={v}")
    return "Network traffic flow. "+", ".join(parts)+" . Attack type:"
print("feature->text...",flush=True); texts_all=df.apply(row_to_text,axis=1).values
print(f"{DATASET}: feature={len(feats)} class={len(classes_all)} test={len(te_idx):,} | seeds={SEEDS}",flush=True)

def balanced_train(seed):
    rng=np.random.default_rng(seed); sel=[]
    for cls in pd.unique(y_all[tr_idx]):
        ids=tr_idx[y_all[tr_idx]==cls]
        if len(ids)>TRAIN_CAP: ids=rng.choice(ids,TRAIN_CAP,replace=False)
        sel+=ids.tolist()
    return np.array(sorted(sel))

def scores(yt,pred): return round(accuracy_score(yt,pred),4), round(f1_score(yt,pred,average="macro",labels=classes_all,zero_division=0),4)
yte=y_all[te_idx]

def run_llm(tr_sel,seed):
    torch.manual_seed(seed); np.random.seed(seed)
    le=LabelEncoder().fit(y_all[tr_sel]); K=len(le.classes_)
    tok=AutoTokenizer.from_pretrained(MODEL)
    if tok.pad_token is None: tok.pad_token=tok.eos_token
    class DS(torch.utils.data.Dataset):
        def __init__(s,idx): s.idx=idx
        def __len__(s): return len(s.idx)
        def __getitem__(s,i):
            j=s.idx[i]; e=tok(texts_all[j],truncation=True,max_length=MAX_LEN,padding="max_length",return_tensors="pt")
            it={k:v.squeeze(0) for k,v in e.items()}; it["labels"]=torch.tensor(int(le.transform([y_all[j]])[0])); return it
    bnb=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type="nf4",bnb_4bit_compute_dtype=torch.bfloat16,bnb_4bit_use_double_quant=True)
    base=AutoModelForSequenceClassification.from_pretrained(MODEL,num_labels=K,quantization_config=bnb,device_map={"":0})
    base=prepare_model_for_kbit_training(base); base.config.pad_token_id=tok.pad_token_id
    model=get_peft_model(base,LoraConfig(task_type=TaskType.SEQ_CLS,r=16,lora_alpha=32,lora_dropout=0.05,
        target_modules=["q_proj","v_proj"],modules_to_save=["score"])); model.config.use_cache=False
    args=TrainingArguments(output_dir=str(ROOT/"models"/"tmp_ms"),per_device_train_batch_size=8,gradient_accumulation_steps=2,
        num_train_epochs=EPOCHS,learning_rate=2e-4,bf16=BF16,fp16=(not BF16),gradient_checkpointing=True,
        logging_steps=100,save_strategy="no",report_to=[],seed=seed)
    Trainer(model=model,args=args,train_dataset=DS(tr_sel)).train()
    dev=next(model.parameters()).device; model.eval(); preds=[]; bs=64
    for s in range(0,len(te_idx),bs):
        js=te_idx[s:s+bs]
        enc=tok(list(texts_all[js]),truncation=True,max_length=MAX_LEN,padding=True,return_tensors="pt").to(dev)
        with torch.no_grad(), torch.autocast(device_type="cuda",dtype=torch.bfloat16,enabled=(dev.type=="cuda")):
            preds+=model(**enc).logits.argmax(-1).cpu().tolist()
    p=le.inverse_transform(np.array(preds))
    del model,base; gc.collect(); torch.cuda.empty_cache()
    return scores(yte,p)

OUT=RES/"multiseed_report.json"
allm=json.load(open(OUT)) if OUT.exists() else {}
allm.setdefault(DATASET,{"model":MODEL,"train_cap":TRAIN_CAP,"epochs":EPOCHS,"seeds":{}})
S=allm[DATASET]["seeds"]
for seed in SEEDS:
    k=str(seed)
    if k in S and all(m in S[k] for m in ["llm","rf","xgb"]):
        print(f"seed {seed} already exists, skipping",flush=True); continue
    t0=time.time(); tr_sel=balanced_train(seed)
    print(f"\n=== seed {seed} | train={len(tr_sel):,} ===",flush=True)
    rf=RandomForestClassifier(n_estimators=300,n_jobs=-1,class_weight="balanced",random_state=seed).fit(X[tr_sel],y_all[tr_sel])
    rf_acc,rf_f1=scores(yte,rf.predict(X[te_idx])); print(f"  RF : acc={rf_acc} macroF1={rf_f1}",flush=True)
    le2=LabelEncoder().fit(y_all[tr_sel])
    xg=xgb.XGBClassifier(n_estimators=300,max_depth=8,n_jobs=-1,tree_method="hist",random_state=seed,
        num_class=len(le2.classes_),objective="multi:softmax",eval_metric="mlogloss").fit(X[tr_sel],le2.transform(y_all[tr_sel]))
    xg_acc,xg_f1=scores(yte,le2.inverse_transform(xg.predict(X[te_idx]))); print(f"  XGB: acc={xg_acc} macroF1={xg_f1}",flush=True)
    llm_acc,llm_f1=run_llm(tr_sel,seed); print(f"  LLM: acc={llm_acc} macroF1={llm_f1}",flush=True)
    S[k]={"llm":{"acc":llm_acc,"macro_f1":llm_f1},"rf":{"acc":rf_acc,"macro_f1":rf_f1},"xgb":{"acc":xg_acc,"macro_f1":xg_f1},"sec":round(time.time()-t0,0)}
    json.dump(allm,open(OUT,"w"),indent=2,ensure_ascii=False)
    print(f"  saved ({time.time()-t0:.0f}s) -> results/multiseed_report.json",flush=True)

# --- OZET (ortalama +/- std) ---
def ms(key):
    v=[S[k][key]["macro_f1"] for k in S]; return np.mean(v),np.std(v),v
print("\n================ COK-SEED OZET (Edge macro-F1) ================")
for key in ["llm","rf","xgb"]:
    m,s,v=ms(key); print(f"  {key:4s}  {m:.4f} +/- {s:.4f}   (seedler: {[f'{x:.3f}' for x in v]})")
lm,ls,_=ms("llm"); rm,rs,_=ms("rf")
print(f"\n  LLM - RF (macro-F1 fark) = {lm-rm:+.4f}   | LLM std={ls:.4f} RF std={rs:.4f}")
print("  -> fark std'lerden buyukse anlamli; degilse ~berabere (durust raporla).")
print("DONE.")
